# Anna의 Supabase DB 연결과 테이블 설계

이 노트북은 팀 공통 `DatabaseManager`로 Supabase PostgreSQL에 연결하고, **anna_rag 전용 스키마에 테이블 5개를 생성**합니다. 각 셀을 위에서부터 실행하세요.

실제 그림 파일은 기존 **images 버킷의 cars/hyundai/ioniq5/** 아래에 저장합니다. DB에는 그림 설명과 파일 위치를 기록합니다. 이 노트북은 그림 업로드, PDF 전체 청킹, 임베딩 API 호출을 하지 않습니다.

생성 셀을 실행하면 원격 DB에 빈 테이블이 생성됩니다. 기존 public 테이블은 변경하지 않습니다. 재실행해도 기존 테이블을 삭제하지 않으며, 이미 있는 테이블의 컬럼 구조를 자동 변경하지도 않습니다.


## 수업 내용과 비교해서 읽기

학습 폴더의 `9_관계형DB_pgvector`와 `10_RAG_에이전트` 노트북을 기준으로 비교했습니다. **기본 도구는 수업과 같지만, 현재 DB 설계에는 수업 예제보다 어려운 추가 규칙이 들어 있습니다.**

| 수업에서 배운 내용 | 이 노트북에서 사용하는 곳 |
|---|---|
| `psycopg.connect(DB_URL)`로 PostgreSQL 연결 | 팀의 `DatabaseManager.connect()`가 내부에서 같은 연결 도구를 사용합니다. 클래스 이름은 팀이 만든 것입니다. |
| `CREATE TABLE`, `PRIMARY KEY`, `FOREIGN KEY` | 설명서·소제목·그림·청크를 저장하고 서로 연결합니다. |
| `SELECT`, `COUNT`, `JOIN` | 만들어진 테이블과 저장 건수를 확인합니다. |
| `conn.commit()`으로 변경 확정 | 여기서는 `with db.connect(...) as conn:`이 성공 시 확정, 실패 시 취소와 연결 종료를 처리합니다. |
| pgvector의 `vector(1536)` | 현재는 차원 미정인 `vector`로 만들어 두었습니다. 수업 방식과 다른 부분입니다. |

### 제가 프로젝트용으로 추가한 내용

- **`anna_rag` 스키마:** 같은 DB 안에서 내 테이블을 별도 폴더처럼 묶습니다.
- **복합 키:** 문서 ID와 소제목 ID를 함께 확인합니다. 다른 PDF에 같은 소제목 ID가 있어도 구분하기 위한 규칙입니다.
- **`section_images`:** 소제목과 이미지의 연결을 별도 테이블에 기록합니다.
- **배열·CHECK·파일 해시:** 여러 페이지 기록, 잘못된 값 검사, 같은 PDF 구별에 사용합니다. 수업의 단순 테이블 예제보다 확장된 설계입니다.
- **`sql.Identifier`, 접속 대상 검사, 제한 시간:** SQL 이름 조합과 실행 관리에 쓰는 보조 코드입니다. RAG 검색 원리를 배우기 위한 핵심은 아닙니다.
- **시스템 테이블 조회:** PostgreSQL이 보관하는 테이블 구조 설명을 읽는 코드입니다. 차량 설명서 본문을 검색하는 코드와 구분하세요.

### 임베딩과 검색은 수업 코드와 아직 연결되지 않았습니다

수업의 `10_RAG_에이전트/1_pgvector_langchain.ipynb`에서는 `OpenAIEmbeddings(model="text-embedding-3-small")`, `PGEngine`, `PGVectorStore.create_sync()`를 사용했습니다. 현재 노트북은 **벡터를 넣을 자리만 준비**하며, 임베딩 생성과 검색 코드는 없습니다.

수업 방식으로 이어가려면 임베딩 모델과 차원을 정하고, `PGVectorStore`가 사용할 스키마·ID·본문·벡터 컬럼을 우리 테이블에 맞춰 연결해야 합니다. 수업 예제를 그대로 복사하는 것만으로 이 테이블과 자동 연결되지는 않습니다. 이미지 검색에도 그림 자체가 아닌 사람이 작성한 캡션 텍스트를 사용합니다.

읽을 때는 먼저 **연결 → 테이블별 한 행의 의미 → 기본키와 외래키 → 본문과 벡터 저장 위치**를 이해하세요. 긴 검사 조건과 시스템 조회는 그다음에 읽어도 됩니다.


## 1 저장 구조 이해하기

```text
Supabase Database
└── anna_rag
    ├── manuals           PDF 한 개의 정보
    ├── sections          소제목의 전체 본문과 주의사항
    ├── images            캡션과 Storage 위치
    ├── section_images    소제목과 그림의 연결
    └── chunks            검색용 글과 임베딩 벡터

Supabase Storage
└── images / cars/hyundai/ioniq5/실제이미지.png
```

설명서 한 개에는 소제목 여러 개, 소제목 하나에는 청크와 그림 여러 개가 들어갑니다. 같은 그림을 여러 소제목에서 참조할 수도 있어 연결 테이블을 둡니다.

현재 청킹 노트북의 `source_id`는 여기의 `document_id`, `parent_id`는 `section_id`에 대응합니다. 이미지 캡션은 images에 보관하고, 검색용 제목+캡션은 이미지 종류의 chunks 행에 넣습니다. 캡션을 바꾸면 검색용 본문과 벡터도 함께 갱신해야 합니다.


## 2 도구와 프로젝트 경로 준비

프로젝트 `.venv` 커널을 선택하세요. DB 접속 정보는 프로젝트 루트의 `.env`에 있는 `DB_URL`을 사용합니다. 비밀번호나 전체 접속 주소를 노트북에 적거나 출력하지 않습니다.

Storage 업로드의 `SUPABASE_URL`·`SUPABASE_SECRET_KEY`와 PostgreSQL 연결용 `DB_URL`은 다른 설정입니다.


In [ ]:
# 경로와 환경변수를 다룰 기본 도구입니다.
from pathlib import Path
import os
import sys

# .env를 읽고, SQL 식별자를 안전하게 처리하며, 조회 결과를 표로 보여줍니다.
from dotenv import load_dotenv
from psycopg import sql
from psycopg.conninfo import conninfo_to_dict
import pandas as pd
from IPython.display import display

# 실제 프로젝트 루트입니다. 다른 컴퓨터에서는 이 경로만 수정합니다.
project_root = Path("/Users/anna/mle-02-p1-team1")
src_dir = project_root / "src"
# 팀 공통 모듈을 찾을 수 있도록 src를 Python 검색 경로에 추가합니다.
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
from car_search_rag.common.database_manager import DatabaseManager

# 명시한 프로젝트의 .env를 읽습니다. 기존 환경변수는 덮어쓰지 않습니다.
load_dotenv(project_root / ".env", override=False)
# 필수 설정의 존재만 확인하고 값은 출력하지 않습니다.
if not os.getenv("DB_URL"):
    raise ValueError("프로젝트 .env에 DB_URL을 설정하세요.")

# 실수로 다른 프로젝트에 테이블을 만들지 않도록 접속 대상을 확인합니다.
connection_info = conninfo_to_dict(os.environ["DB_URL"])
project_ref = "hbuvewommtcmbbynsvht"
if (project_ref not in connection_info.get("host", "")
        and project_ref not in connection_info.get("user", "")):
    raise ValueError("DB_URL이 팀 Supabase 프로젝트를 가리키는지 확인하세요.")

# 접속 대기와 쿼리 대기에 기본 제한을 둡니다. 공통 코드 파일은 바꾸지 않습니다.
os.environ.setdefault("PGCONNECT_TIMEOUT", "10")
# 공통 연결 관리 객체를 만듭니다. 실제 접속은 connect()에서 이루어집니다.
db = DatabaseManager()
print("DB 연결 준비 완료: 접속 정보는 출력하지 않습니다.")


## 3 실제 연결과 pgvector 확인

`with` 블록은 작업이 성공하면 커밋하고, 실패하면 롤백한 뒤 연결을 닫습니다. 여기서는 읽기만 합니다. `camel_case_keys=False`는 조회 결과의 열 이름을 DB 그대로 받는 설정입니다.

벡터 확장은 이미 설치된 것을 사용합니다. 다른 팀의 확장을 새로 만들거나 이동하지 않습니다.


In [ ]:
# DB 연결을 열고 블록이 끝나면 자동으로 닫습니다.
with db.connect(camel_case_keys=False) as conn:
    # 읽기 전용으로 상태만 확인합니다.
    conn.execute("SET TRANSACTION READ ONLY")
    conn.execute("SET LOCAL statement_timeout = '10s'")
    # 설치된 vector 확장의 스키마와 버전을 읽습니다.
    vector_info = conn.execute("""
        SELECT n.nspname AS vector_schema, e.extversion AS vector_version
        FROM pg_extension e
        JOIN pg_namespace n ON n.oid = e.extnamespace
        WHERE e.extname = 'vector'
    """).fetchone()
    if vector_info is None:
        raise RuntimeError("pgvector 설치 상태를 팀 관리자와 확인하세요.")
    # 벡터 자료형이 실제로 설치된 위치를 이후 SQL에 사용합니다.
    vector_schema = vector_info["vector_schema"]
    print("DB 연결 성공 / pgvector:", vector_info["vector_version"])
    # Anna 스키마에 이미 있는 테이블만 조회합니다.
    existing_tables = conn.execute("""
        SELECT table_name FROM information_schema.tables
        WHERE table_schema = 'anna_rag' AND table_type = 'BASE TABLE'
        ORDER BY table_name
    """).fetchall()
    print("현재 Anna 테이블:", [row["table_name"] for row in existing_tables])


## 4 컬럼과 연결 규칙

| 테이블 | 주요 컬럼 | 한 행의 의미 |
|---|---|---|
| manuals | document_id, vehicle_id, model_year, document_version, source_file, file_sha256 | 설명서 한 버전 |
| sections | document_id, section_id, title, section_path, full_text, pdf_pages | 소제목 전체 설명 |
| images | document_id, image_id, caption, pdf_page, storage_bucket, storage_path | 그림 한 장 |
| section_images | document_id, section_id, image_id | 소제목과 그림의 연결 한 개 |
| chunks | chunk_id, document_id, section_id, chunk_type, page_content, image_id, embedding | 검색 자료 한 개 |

- `PRIMARY KEY`: 행의 고유 이름입니다. 같은 키를 중복 INSERT하면 오류가 납니다.
- `FOREIGN KEY`: 존재하는 상위 자료만 연결하도록 하는 규칙입니다. 다른 PDF의 소제목·이미지가 섞이지 않게 document_id도 함께 검사합니다.
- `NOT NULL`: 반드시 입력합니다. `NULL`은 아직 값이 없음을 뜻합니다.
- `CHECK`: 허용되는 값인지 검사합니다. 텍스트 청크에는 image_id를 비우고, 이미지 청크에는 실제 연결된 그림 ID를 넣습니다.
- `pdf_pages`: `ARRAY[18, 19]`처럼 여러 페이지를 기록합니다. 페이지가 넘어가도 같은 소제목을 유지할 수 있습니다.
- `printed_pages`: 인쇄 쪽수는 문자열 배열로 기록합니다. 확인하지 못했으면 NULL로 둡니다.
- `file_sha256`: PDF 내용으로 계산한 식별값입니다. 같은 파일을 다시 적재하는 중복을 구별하는 데 사용합니다.

**ID 규칙:** 문서마다 document_id를 새로 정하고, 같은 PDF를 다시 처리할 때는 같은 ID를 사용합니다. section_id와 image_id는 문서 안에서 고유하면 됩니다. chunk_id는 문서 ID를 포함해 전체에서 고유하게 만듭니다. DB가 안정적인 ID를 대신 만들어 주지는 않습니다.

**재적재:** 이 노트북은 빈 테이블만 생성합니다. 실제 적재 코드는 다음 단계에서 키 기준 INSERT 또는 UPSERT를 구현합니다. 청킹 규칙이 바뀌어 이전 청크가 남는 문제는 별도의 재적재 처리로 해결해야 합니다.


## 5 임베딩 컬럼 설계

아직 임베딩 모델을 확정하지 않았으므로 `embedding`은 차원을 고정하지 않은 `vector` 자료형으로 준비합니다. 처음 청킹 결과를 넣을 때는 NULL로 두고, 임베딩 후 벡터·모델·차원을 함께 저장합니다.

`embedding_model`에는 모델과 필요한 버전 구분을 기록합니다. `embedding_dimension`은 숫자 개수입니다. DB는 기록된 차원이 실제 벡터 길이와 같은지 검사합니다. 벡터가 NULL일 때 모델·차원도 NULL로 두는 규칙입니다.

**같은 차원이어도 다른 모델의 벡터를 서로 비교하면 안 됩니다.** 실제 검색은 모델과 차원을 제한하도록 구현해야 합니다. 모델 확정 뒤에는 고정 차원 컬럼 또는 모델별 검색 인덱스를 검토합니다. 현재는 벡터 검색 인덱스를 만들지 않습니다.

참고: [pgvector 공식 문서](https://github.com/pgvector/pgvector)


## 6 테이블 생성 SQL 준비

아래 셀은 SQL 문자열을 준비할 뿐, 아직 DB를 변경하지 않습니다. `--`로 시작하는 부분은 SQL 주석입니다. Python의 `#` 주석과 역할은 같지만 언어가 다릅니다.

SQL 안의 `{vector_schema}`에는 3번에서 확인한 확장 스키마 이름을 안전하게 넣습니다. 모든 테이블은 `anna_rag.`로 명시하여 public과 구분합니다.


In [ ]:
# DDL은 테이블 구조를 정의하는 SQL입니다. 여기서는 문자열로만 보관합니다.
ddl = """
-- 개인 작업용 스키마입니다. 존재하면 그대로 사용합니다.
CREATE SCHEMA IF NOT EXISTS anna_rag;

-- 1. 설명서 한 버전의 정보
CREATE TABLE IF NOT EXISTS anna_rag.manuals (
    document_id text PRIMARY KEY,                         -- 문서 ID
    vehicle_id text NOT NULL,                             -- 차종 ID
    model_year integer CHECK (model_year BETWEEN 1900 AND 2200), -- 확인한 연식
    document_version text NOT NULL,                       -- 설명서 버전
    source_file text NOT NULL,                            -- 원본 파일명 또는 위치
    file_sha256 text NOT NULL UNIQUE                      -- 원본 중복 방지용 해시
        CHECK (file_sha256 ~ '^[0-9a-f]{64}$'),
    created_at timestamptz NOT NULL DEFAULT now()          -- 생성 시각
);

-- 2. 전체 설명을 보관하는 부모 문서
CREATE TABLE IF NOT EXISTS anna_rag.sections (
    document_id text NOT NULL REFERENCES anna_rag.manuals(document_id),
    section_id text NOT NULL,                             -- 기존 parent_id
    title text NOT NULL,                                  -- 현재 소제목
    section_path text[] NOT NULL,                         -- 상위 목차부터 현재 제목까지
    full_text text NOT NULL CHECK (length(trim(full_text)) > 0),
    pdf_pages integer[] NOT NULL                          -- 여러 페이지에 걸칠 수 있음
        CHECK (cardinality(pdf_pages) > 0 AND 0 < ALL(pdf_pages)
               AND array_position(pdf_pages, NULL) IS NULL),
    printed_pages text[],                                -- 확인 가능한 인쇄 쪽수
    section_order integer NOT NULL CHECK (section_order > 0),
    PRIMARY KEY (document_id, section_id),                -- 문서 안에서 소제목 ID 고유
    UNIQUE (document_id, section_order)
);

-- 3. 그림 파일 자체는 Storage에, 여기에는 설명과 위치를 저장
CREATE TABLE IF NOT EXISTS anna_rag.images (
    document_id text NOT NULL REFERENCES anna_rag.manuals(document_id),
    image_id text NOT NULL,
    caption text,                                        -- 캡션은 나중에 작성 가능
    caption_source text NOT NULL DEFAULT 'manual',
    pdf_page integer NOT NULL CHECK (pdf_page > 0),
    storage_bucket text NOT NULL DEFAULT 'images'
        CHECK (storage_bucket = 'images'),
    storage_path text NOT NULL                            -- 버킷명 제외한 객체 경로
        CHECK (storage_path LIKE 'cars/hyundai/ioniq5/%'),
    PRIMARY KEY (document_id, image_id),
    UNIQUE (storage_bucket, storage_path)                 -- 같은 파일의 중복 등록 방지
);

-- 4. 소제목 하나에 여러 그림, 그림 하나에 여러 소제목 연결 가능
CREATE TABLE IF NOT EXISTS anna_rag.section_images (
    document_id text NOT NULL,
    section_id text NOT NULL,
    image_id text NOT NULL,
    PRIMARY KEY (document_id, section_id, image_id),
    FOREIGN KEY (document_id, section_id)
        REFERENCES anna_rag.sections(document_id, section_id),
    FOREIGN KEY (document_id, image_id)
        REFERENCES anna_rag.images(document_id, image_id)
);

-- 5. 검색용 텍스트와 그 텍스트의 임베딩을 같은 행에 저장
CREATE TABLE IF NOT EXISTS anna_rag.chunks (
    chunk_id text PRIMARY KEY,
    document_id text NOT NULL,
    section_id text NOT NULL,
    chunk_type text NOT NULL CHECK (chunk_type IN ('text', 'image')),
    chunk_order integer NOT NULL CHECK (chunk_order > 0),
    page_content text NOT NULL CHECK (length(trim(page_content)) > 0),
    pdf_pages integer[] NOT NULL
        CHECK (cardinality(pdf_pages) > 0 AND 0 < ALL(pdf_pages)
               AND array_position(pdf_pages, NULL) IS NULL),
    image_id text,                                       -- 텍스트 청크에는 NULL
    embedding {vector_schema}.vector,                    -- 모델 미정이므로 차원은 추후 확정
    embedding_model text,                                -- 모델/버전 식별값
    embedding_dimension integer CHECK (embedding_dimension > 0),
    created_at timestamptz NOT NULL DEFAULT now(),
    FOREIGN KEY (document_id, section_id)
        REFERENCES anna_rag.sections(document_id, section_id),
    FOREIGN KEY (document_id, section_id, image_id)
        REFERENCES anna_rag.section_images(document_id, section_id, image_id),
    UNIQUE (document_id, section_id, chunk_type, chunk_order),
    CHECK ((chunk_type = 'text' AND image_id IS NULL)
        OR (chunk_type = 'image' AND image_id IS NOT NULL)),
    CHECK (
        (embedding IS NULL AND embedding_model IS NULL AND embedding_dimension IS NULL)
        OR
        (embedding IS NOT NULL AND embedding_model IS NOT NULL
         AND length(trim(embedding_model)) > 0 AND embedding_dimension IS NOT NULL
         AND {vector_schema}.vector_dims(embedding) = embedding_dimension)
    )
);

-- 연결 ID를 찾기 위한 일반 인덱스입니다. 벡터 검색용 인덱스는 아닙니다.
CREATE INDEX IF NOT EXISTS chunks_section_idx
    ON anna_rag.chunks(document_id, section_id);
CREATE INDEX IF NOT EXISTS section_images_image_idx
    ON anna_rag.section_images(document_id, image_id);
CREATE UNIQUE INDEX IF NOT EXISTS chunks_one_image_per_section_idx
    ON anna_rag.chunks(document_id, section_id, image_id)
    WHERE chunk_type = 'image';
"""
# SQL 자료형의 스키마 이름을 안전한 식별자로 처리합니다.
# 정규표현식의 {64}가 SQL 템플릿의 치환 대상으로 해석되지 않게 합니다.
create_tables_sql = sql.SQL(ddl.replace("{64}", "{{64}}" )).format(
    vector_schema=sql.Identifier(vector_schema)
)
print("Anna 전용 테이블 5개의 SQL을 준비했습니다. 다음 셀에서 실행합니다.")


## 7 스키마와 테이블 생성

**이 셀은 Supabase DB에 실제로 구조를 생성합니다.** 빈 테이블과 일반 인덱스만 만들고 설명서 데이터는 아직 넣지 않습니다.

처리를 하나의 트랜잭션으로 실행합니다. 중간에 실패하면 이번 변경을 함께 취소합니다. IF NOT EXISTS는 중복 생성을 피하지만 이미 존재하는 테이블의 구조가 달라도 자동으로 수정하지 않습니다.


In [ ]:
# 같은 연결의 생성 작업은 성공했을 때 함께 확정됩니다.
with db.connect(camel_case_keys=False) as conn:
    # 다른 작업의 잠금 때문에 오래 기다리지 않도록 제한합니다.
    conn.execute("SET LOCAL lock_timeout = '5s'")
    conn.execute("SET LOCAL statement_timeout = '30s'")
    # 6번에서 준비한 SQL을 실제 DB에서 실행합니다.
    conn.execute(create_tables_sql)
print("anna_rag 테이블 생성 완료. 기존 테이블과 데이터는 유지됩니다.")


## 8 실제 컬럼과 연결 관계 확인

테이블 이름뿐 아니라 컬럼의 자료형과 제약조건을 확인합니다. Supabase Database의 스키마 선택에서 **anna_rag**를 선택하면 생성된 테이블이 보입니다.

현재 방식은 PostgreSQL 직접 연결입니다. Supabase Data API에 스키마를 공개하는 설정은 하지 않습니다. 스키마는 정리용 구분이며, 같은 DB 권한을 가진 팀원과 접근 권한까지 완전히 분리하는 것은 아닙니다.


In [ ]:
# 읽기 전용으로 컬럼과 제약조건 목록을 조회합니다.
with db.connect(camel_case_keys=False) as conn:
    conn.execute("SET TRANSACTION READ ONLY")
    columns = conn.execute("""
        SELECT table_name, column_name, data_type, udt_name, is_nullable
        FROM information_schema.columns
        WHERE table_schema = 'anna_rag'
        ORDER BY table_name, ordinal_position
    """).fetchall()
    constraints = conn.execute("""
        SELECT c.relname AS table_name, con.conname AS constraint_name,
               pg_get_constraintdef(con.oid) AS rule
        FROM pg_constraint con
        JOIN pg_class c ON c.oid = con.conrelid
        JOIN pg_namespace n ON n.oid = c.relnamespace
        WHERE n.nspname = 'anna_rag'
        ORDER BY c.relname, con.conname
    """).fetchall()
# 표로 확인합니다. 긴 설명과 많은 행을 생략하지 않도록 표시합니다.
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(pd.DataFrame(columns))
    display(pd.DataFrame(constraints))


## 9 저장된 데이터 건수 확인

처음에는 모두 0건입니다. 이후 PDF 처리로 데이터를 넣으면 이 셀을 다시 실행해 건수를 확인합니다.


In [ ]:
# 건수를 확인할 테이블 이름을 리스트로 준비합니다.
table_names = ["manuals", "sections", "images", "section_images", "chunks"]
counts = []
with db.connect(camel_case_keys=False) as conn:
    conn.execute("SET TRANSACTION READ ONLY")
    for table_name in table_names:
        # 테이블 이름은 SQL 식별자로 조합합니다.
        query = sql.SQL("SELECT count(*) AS count FROM {}.{}").format(
            sql.Identifier("anna_rag"), sql.Identifier(table_name)
        )
        count = conn.execute(query).fetchone()["count"]
        counts.append({"테이블": table_name, "건수": count})
display(pd.DataFrame(counts))


## 10 다음 노트북으로 이어가기

이번에는 저장할 테이블만 만들었습니다. 다음에는 소제목 몇 개를 아래 규칙으로 저장하고 확인한 뒤 전체 PDF로 확대합니다.

| 2번 노트북의 값 | 이번 DB의 저장 위치 |
|---|---|
| source_id | manuals.document_id. 개정판은 별도 버전 ID 사용 |
| parent_id | sections.section_id |
| parent_doc.page_content | sections.full_text |
| title | sections.title |
| 원본에서 확인한 상위 목차 | sections.section_path |
| chunk.page_content | chunks.page_content |
| chunk.metadata의 chunk_id | chunks.chunk_id |
| image_id | images.image_id |
| 업로드한 이미지의 객체 경로 | images.storage_path |
| 같은 소제목과 이미지의 관계 | section_images |

**등록 순서:** manuals → sections와 images → section_images → chunks. 상위 자료가 먼저 있어야 합니다.

**Storage:** 버킷은 `images`, 경로는 `cars/hyundai/ioniq5/파일명.png`입니다. 공개 URL이나 만료되는 URL 대신 버킷과 객체 경로를 기록합니다. DB 등록만으로 그림이 업로드되지는 않습니다.

**검색 대상:** 텍스트 본문과 수동 이미지 캡션입니다. 이미지 자체 임베딩과 AI 이미지 해석은 하지 않습니다.

**임베딩 전:** embedding, embedding_model, embedding_dimension은 모두 NULL로 둡니다.

**임베딩 후:** 세 항목을 함께 갱신합니다. 본문이 바뀌면 기존 벡터를 유지하지 말고 세 항목을 NULL로 바꾼 뒤 다시 임베딩합니다.

전체 PDF의 목차 구분, 청킹과 이미지 연결, 중복 없이 적재하는 코드, 임베딩 생성과 검색은 다음 단계에서 구현합니다.

참고: [Supabase의 커스텀 스키마](https://supabase.com/docs/guides/api/using-custom-schemas)
